# ASTE Model v3 — Research Paper-Inspired Approach

**Based on:** *"Multilingual sentiment analysis in restaurant reviews using aspect focused learning"*
(Rahman et al., Scientific Reports 2025, XLM-RSA model)

## Key techniques adapted from the paper:
1. **FLAN-T5-base** backbone — instruction-tuned T5 (better at generation tasks than vanilla T5)
2. **Text preprocessing pipeline** — noise reduction (remove special chars, normalize whitespace) as in paper Sec. 4
3. **Training config** from paper: AdamW, lr=2e-5, weight_decay=0.01, batch_size=32, 10 epochs
4. **Label smoothing** (0.1) — reduces overconfidence, improves generalization
5. **Richer prompt template** — includes domain keyword (REST/LAP) for aspect-focused context
6. **Category is ignored** as instructed

## 0. Install Dependencies

In [ ]:
!pip install transformers datasets sentencepiece protobuf accelerate -q

In [ ]:
import json
import re
import os
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration (Paper-Inspired)

In [ ]:
# ── Paths (Kaggle) ──
DATA_DIR       = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE      = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE       = DATA_DIR / "laptop_train.jsonl"
MODEL_DIR      = Path("/kaggle/working/aste_flant5_model")
OUTPUT_FILE    = Path("/kaggle/working/predictions.jsonl")

# ── Model ──
MODEL_NAME       = "google/flan-t5-base"          # instruction-tuned T5 (250M params)

# ── Hyper-parameters (from research paper) ──
MAX_INPUT_LEN    = 256
MAX_TARGET_LEN   = 256
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 4                              # effective batch = 32 (paper uses 32)
LEARNING_RATE    = 2e-5                            # paper: 2e-5
WEIGHT_DECAY     = 0.01                            # paper: 1e-2
NUM_EPOCHS       = 15
WARMUP_RATIO     = 0.1
LABEL_SMOOTHING  = 0.1                             # reduces overconfidence
VAL_SPLIT        = 0.1
LOGGING_STEPS    = 50
NUM_BEAMS        = 5
LENGTH_PENALTY   = 1.0

## 2. Data Preprocessing (Paper-Inspired)

From the paper's methodology (Section 4):
- **Noise reduction**: Remove irrelevant symbols, special characters, extra whitespace
- **Domain-aware prompts**: Include REST/LAP keyword for aspect-focused learning

In [ ]:
def preprocess_text(text):
    """
    Noise reduction as described in the research paper.
    Clean and normalize text while preserving case (required for output).
    """
    # Remove non-informative tokens (paper: Sclean = Clean(S))
    text = re.sub(r'[\t\n\r\f\v]+', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Strip
    text = text.strip()
    return text


def load_jsonl(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


rest_data = load_jsonl(REST_FILE)
lap_data  = load_jsonl(LAP_FILE)

# Add domain keyword (REST/LAP) for aspect-focused context
for r in rest_data:
    r["domain"] = "restaurant"
for r in lap_data:
    r["domain"] = "laptop"

all_data = rest_data + lap_data

print(f"Restaurant: {len(rest_data)}")
print(f"Laptop:     {len(lap_data)}")
print(f"Total:      {len(all_data)}")

### 2.1 Linearize Triplets with Domain-Aware Prompt

- **Input**: `"Extract aspect sentiment triplets from this <domain> review: <text>"`
- **Target**: `"( Aspect | Opinion | V#A ) ; ..."`

In [ ]:
def record_to_seq2seq(record):
    """
    Domain-aware prompt for FLAN-T5 (instruction-tuned model).
    """
    text = preprocess_text(record["Text"])
    domain = record.get("domain", "")
    input_text = f"Extract aspect sentiment triplets from this {domain} review: {text}"

    triplet_strs = []
    for quad in record["Quadruplet"]:
        aspect  = quad.get("Aspect", "NULL")
        opinion = quad.get("Opinion", "NULL")
        va      = quad.get("VA", "5.00#5.00")
        triplet_strs.append(f"( {aspect} | {opinion} | {va} )")

    target_text = " ; ".join(triplet_strs)
    return input_text, target_text


pairs = []
for rec in all_data:
    inp, tgt = record_to_seq2seq(rec)
    pairs.append({"id": rec["ID"], "input": inp, "target": tgt, "domain": rec.get("domain", "")})

print(f"Total seq2seq pairs: {len(pairs)}")
print(f"\nExample (restaurant):")
ex_r = next(p for p in pairs if p['domain'] == 'restaurant')
print(f"  INPUT:  {ex_r['input'][:130]}...")
print(f"  TARGET: {ex_r['target'][:120]}...")
print(f"\nExample (laptop):")
ex_l = next(p for p in pairs if p['domain'] == 'laptop')
print(f"  INPUT:  {ex_l['input'][:130]}...")
print(f"  TARGET: {ex_l['target'][:120]}...")

### 2.2 Train / Validation Split

In [ ]:
random.shuffle(pairs)
split_idx = int(len(pairs) * (1 - VAL_SPLIT))
train_pairs = pairs[:split_idx]
val_pairs   = pairs[split_idx:]

print(f"Train: {len(train_pairs)}")
print(f"Val:   {len(val_pairs)}")

## 3. Tokenization & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ASTEDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input_len, max_target_len):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_enc = self.tokenizer(
            pair["input"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        target_enc = self.tokenizer(
            pair["target"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = target_enc["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels":         labels,
        }

train_dataset = ASTEDataset(train_pairs, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_dataset   = ASTEDataset(val_pairs,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
sample = train_dataset[0]
print(f"input_ids: {sample['input_ids'].shape} | labels: {sample['labels'].shape}")

## 4. Model & Training (FLAN-T5-base + Paper Config)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,                    # paper: 2e-5
    weight_decay=WEIGHT_DECAY,                      # paper: 0.01
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    label_smoothing_factor=LABEL_SMOOTHING,          # reduces overconfidence
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=3,
    eval_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
total_steps = (len(train_dataset) // eff_batch) * NUM_EPOCHS

print("Trainer ready (Research Paper Configuration)")
print(f"  Model:            {MODEL_NAME}")
print(f"  Parameters:       {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device:           {DEVICE}  |  FP16: {training_args.fp16}")
print(f"  Epochs:           {NUM_EPOCHS}")
print(f"  Batch (eff):      {eff_batch} ({BATCH_SIZE} × {GRAD_ACCUM_STEPS})")
print(f"  LR:               {LEARNING_RATE} (cosine, paper: 2e-5)")
print(f"  Weight decay:     {WEIGHT_DECAY} (paper: 0.01)")
print(f"  Label smoothing:  {LABEL_SMOOTHING}")
print(f"  Total steps:      {total_steps}")

In [ ]:
# ══════════════════════════════════════════════════════
#  TRAINING  (Tesla T4: ~45-90 min)
# ══════════════════════════════════════════════════════
trainer.train()
print("\n✅ Training complete!")

In [ ]:
trainer.save_model(str(MODEL_DIR / "best"))
tokenizer.save_pretrained(str(MODEL_DIR / "best"))
print(f"Model saved to: {MODEL_DIR / 'best'}")

## 5. Inference

In [ ]:
def parse_triplets(generated_text):
    triplets = []
    pattern = r"\(\s*(.+?)\s*\|\s*(.+?)\s*\|\s*([\d.]+#[\d.]+)\s*\)"
    for match in re.finditer(pattern, generated_text):
        aspect  = match.group(1).strip()
        opinion = match.group(2).strip()
        va      = match.group(3).strip()
        try:
            v, a = va.split("#")
            v, a = float(v), float(a)
            v = max(1.0, min(9.0, v))
            a = max(1.0, min(9.0, a))
            va = f"{v:.2f}#{a:.2f}"
        except:
            va = "5.00#5.00"
        triplets.append({"Aspect": aspect, "Opinion": opinion, "VA": va})
    return triplets

print("Parse test:", parse_triplets("( food | great | 7.50#6.00 ) ; ( service | slow | 3.00#5.50 )"))

In [ ]:
def predict_triplets(texts, model, tokenizer, domains=None,
                     max_len=256, batch_size=16, num_beams=5, length_penalty=1.0):
    model.eval()
    all_triplets = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        batch_domains = domains[i : i + batch_size] if domains else ["" for _ in batch_texts]

        inputs = [
            f"Extract aspect sentiment triplets from this {d} review: {preprocess_text(t)}"
            for t, d in zip(batch_texts, batch_domains)
        ]

        encoded = tokenizer(
            inputs, max_length=max_len, padding=True,
            truncation=True, return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded, max_length=max_len,
                num_beams=num_beams, length_penalty=length_penalty,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for gen_text in decoded:
            all_triplets.append(parse_triplets(gen_text))

    return all_triplets

print("Inference function ready.")

## 6. Evaluate on Validation Set

In [ ]:
val_texts   = [p["input"].split("review: ", 1)[-1] for p in val_pairs]
val_domains = [p.get("domain", "") for p in val_pairs]
val_golds   = [p["target"] for p in val_pairs]
val_ids     = [p["id"] for p in val_pairs]

val_preds = predict_triplets(
    val_texts, model, tokenizer, domains=val_domains,
    max_len=MAX_TARGET_LEN, num_beams=NUM_BEAMS, length_penalty=LENGTH_PENALTY,
)

for i in range(min(8, len(val_preds))):
    print(f"\n{'='*70}")
    print(f"ID:    {val_ids[i]}")
    print(f"Text:  {val_texts[i][:100]}...")
    print(f"Gold:  {val_golds[i][:100]}...")
    print(f"Pred:  {val_preds[i]}")

In [ ]:
def compute_metrics(val_pairs, val_preds):
    total_gold = total_pred = correct = 0
    va_errors = []

    for pair, preds in zip(val_pairs, val_preds):
        gold_triplets = parse_triplets(pair["target"])
        total_gold += len(gold_triplets)
        total_pred += len(preds)

        gold_set = {(g["Aspect"].lower(), g["Opinion"].lower()): g["VA"] for g in gold_triplets}
        for p in preds:
            key = (p["Aspect"].lower(), p["Opinion"].lower())
            if key in gold_set:
                correct += 1
                try:
                    gv, ga = gold_set[key].split("#")
                    pv, pa = p["VA"].split("#")
                    va_errors.append(abs(float(gv)-float(pv)) + abs(float(ga)-float(pa)))
                except: pass

    prec = correct / total_pred if total_pred else 0
    rec  = correct / total_gold if total_gold else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0
    avg_va = np.mean(va_errors) if va_errors else float('inf')

    print(f"\n{'='*50}")
    print(f"Span Extraction:")
    print(f"  Gold: {total_gold}  Pred: {total_pred}  Correct: {correct}")
    print(f"  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    print(f"VA Error (matched): {avg_va:.4f}  (n={len(va_errors)})")

compute_metrics(val_pairs, val_preds)

## 7. Generate Test Predictions

In [ ]:
TEST_FILE = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp/test.jsonl")

if TEST_FILE.exists():
    best_path = MODEL_DIR / "best"
    if best_path.exists():
        model = AutoModelForSeq2SeqLM.from_pretrained(str(best_path)).to(DEVICE)
        tokenizer = AutoTokenizer.from_pretrained(str(best_path))
        print(f"Loaded model from: {best_path}")
    else:
        print("Using model in memory.")

    test_data = load_jsonl(TEST_FILE)
    print(f"Test records: {len(test_data)}")

    test_ids   = [r["ID"] for r in test_data]
    test_texts = [r["Text"] for r in test_data]
    # Infer domain from ID prefix
    test_domains = ["restaurant" if r["ID"].startswith("R") or "rest" in r["ID"].lower()
                    else "laptop" for r in test_data]

    test_preds = predict_triplets(
        test_texts, model, tokenizer, domains=test_domains,
        max_len=MAX_TARGET_LEN, num_beams=NUM_BEAMS, length_penalty=LENGTH_PENALTY,
    )

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for rid, triplets in zip(test_ids, test_preds):
            f.write(json.dumps({"ID": rid, "Triplet": triplets}, ensure_ascii=False) + "\n")

    print(f"\n✅ Predictions written to: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, "r") as f:
        for i, line in enumerate(f):
            if i >= 3: break
            print(json.loads(line))
else:
    print(f"⚠️  Test file not found: {TEST_FILE}")

---

## Summary — v3 (Paper-Inspired) vs v1 vs v2

| Feature | v1 | v2 | **v3 (Paper)** |
|---|---|---|---|
| **Model** | T5-small (60M) | T5-base (220M) | **FLAN-T5-base (250M)** |
| **Instruction-tuned** | No | No | **Yes** |
| **Prompt** | `extract triplets:` | `extract triplets:` | **Domain-aware instruction** |
| **LR** | 3e-4 | 2e-4 | **2e-5 (paper)** |
| **Weight decay** | 0.01 | 0.01 | **0.01 (paper)** |
| **Label smoothing** | No | No | **0.1** |
| **Batch (effective)** | 8 | 32 | **32 (paper)** |
| **LR scheduler** | linear | cosine | **cosine** |
| **Text preprocessing** | None | Basic | **Paper-based noise reduction** |

### Paper Reference:
Rahman et al. (2025). *Multilingual sentiment analysis in restaurant reviews using aspect focused learning.*
Scientific Reports 15:28371. DOI: 10.1038/s41598-025-12464-y